In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-21T15:25:28.801827+00:00', 'open': 103.05, 'high': 104.28, 'low': 102.02, 'close': 103.74, 'volume': 433, 'trade_count': 50, 'vwap': 104.14}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-21T15:25:28.801827+00:00', 'open': 98.87, 'high': 102.08, 'low': 98.9, 'close': 100.09, 'volume': 566, 'trade_count': 14, 'vwap': 100.59}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-21T15:25:28.801827+00:00', 'open': 110.97, 'high': 113.18, 'low': 109.81, 'close': 112.32, 'volume': 523, 'trade_count': 44, 'vwap': 111.99}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-21T15:25:29.801992+00:00', 'open': 105.85, 'high': 108.69, 'low': 106.02, 'close': 107.12, 'volume': 917, 'trade_count': 33, 'vwap': 106.7}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-21T15:25:29.801992+00:00', 'open': 97.09, 'high': 99.55, 'low': 95.56, 'close': 97.91, 'volume': 517, 'trade_count': 22, 'vwap': 97.75}
Pushed to